In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.simclr.utils import load_simclr_backbone
from const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/simclr_acdc/resnet-18/checkpoints/epoch=18-step=133.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = load_simclr_backbone(model_path)

In [ ]:
loader_train = data_module.train_dataloader()

# Stack each batch
data_train = []

for batch in loader_train:
    data_train.append(batch)

data_train = torch.cat(data_train, dim=0)
print(data_train.shape)

loader_test = data_module.test_dataloader()

# Stack each batch
data_test = []

for batch in loader_test:
    data_test.append(batch)

data_test = torch.cat(data_test, dim=0)
print(data_test.shape)

In [ ]:
from utils.utils import encode_embeddings

model = model.to(device)

# Extract features for train images
train_feats = encode_embeddings(data_train, model, device)
print(train_feats.shape)

# Extract features for test images
test_feats = encode_embeddings(data_test, model, device)
print(test_feats.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(train_feats)

print(f"Explained variance: {pca.explained_variance_ratio_}")
print(f"Sum: {pca.explained_variance_ratio_.sum()}")

train_feats_reduced = pca.transform(train_feats)
test_feats_reduced = pca.transform(test_feats)

print(train_feats_reduced.shape)
print(test_feats_reduced.shape)

In [ ]:
from matplotlib import pyplot as plt

plt.style.use("ggplot")

plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(test_feats_reduced[:, 0], test_feats_reduced[:, 1], alpha=0.5, s=10, label="Test")

plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
from arch.vae.vae import VAE
from utils.utils import discretise

def generate_data(model: VAE) -> torch.Tensor:
    num_samples = 512

    # Sample from latent space
    z = torch.randn(num_samples, model.hparams.latent_dim).to(device)

    with torch.no_grad():
        model.eval()
        model.to(device)
        
        # Generate segmentation maps from latent variables
        x_fake_logits: torch.Tensor = model.decoder(z)

    return x_fake_logits

vae_model_path = "logs/vae_acdc/info-vae/ld-8-beta-0-gamma-1000/checkpoints/epoch=43-step=4708.ckpt"
vae_model = VAE.load_from_checkpoint(vae_model_path)

good_logits = generate_data(vae_model)

vae_model_path = "logs/vae_acdc/info-vae/ld-8-beta-0-gamma-50/checkpoints/epoch=44-step=4815.ckpt"
vae_model = VAE.load_from_checkpoint(vae_model_path)

bad_logits = generate_data(vae_model)

good_fake_data = discretise(good_logits)
bad_fake_data = discretise(bad_logits)

print(good_fake_data.shape)
print(bad_fake_data.shape)

In [ ]:
from utils.utils import show_samples

# View generations
good_generations = torch.argmax(good_logits, dim=1).unsqueeze(1)[:40]
show_samples(good_generations, rgb=False, ncol=10, figsize=(10, 4))

bad_generations = torch.argmax(bad_logits, dim=1).unsqueeze(1)[:40]
show_samples(bad_generations, rgb=False, ncol=10, figsize=(10, 4))

In [ ]:
good_feats = encode_embeddings(good_fake_data, model, device)
bad_feats = encode_embeddings(bad_fake_data, model, device)

good_feats_reduced = pca.transform(good_feats)
bad_feats_reduced = pca.transform(bad_feats)

print(good_feats_reduced.shape)
print(bad_feats_reduced.shape)

In [ ]:
plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(good_feats_reduced[:, 0], good_feats_reduced[:, 1], alpha=0.5, s=10, label="Good")

plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(bad_feats_reduced[:, 0], bad_feats_reduced[:, 1], alpha=0.5, s=10, label="Bad")

plt.tight_layout()
plt.legend()
plt.show()